# HW5. Olist Customer Features in PostgreSQL: CTE Pipeline, Recursive CTE, Materialized View and EXPLAIN

**Author:** Serhii Kravchenko
**Dataset:** [Brazilian E-Commerce Public Dataset by Olist](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce) (Kaggle, `olistbr/brazilian-ecommerce`), five of its nine CSV files
**Environment:** Google Colab (also runs locally), PostgreSQL 16 via `pgserver`, SQL cells via `jupysql`

The notebook builds customer-level ML features from the Olist data. It loads the CSV files into a typed schema, computes the features with a CTE pipeline of named stages, walks a synthetic referral tree with a recursive CTE, caches the pipeline in a materialized view with concurrent refresh, and compares recomputing the features with reading them from the view using `EXPLAIN (ANALYZE, BUFFERS)`.

Each section drops and recreates the objects it builds, so **Restart session and run all** always starts from a clean state. Query results are the same on every run; only the timings in the `EXPLAIN` output and the `referred_at` timestamps differ.

## 1. Setup and typed schema

### 1.1 Environment

`pgserver` ships PostgreSQL 16 as a Python package, so the database runs inside the notebook session. Its data directory is `hw5_pg` in the system temp folder, which is `/tmp/hw5_pg` on Colab.

A few environment details:
- Colab now defaults to Python 3.13, while `pgserver` 0.1.4 only publishes wheels up to Python 3.12. Its native module is built for the stable ABI (`abi3`), so the 3.12 wheel works on 3.13 once the wheel tag is renamed. The install cell handles this automatically.
- The session time zone is set to UTC, so the `TIMESTAMPTZ` column `referred_at` looks the same on every machine. The Windows build of `pgserver` comes without the time zone database and does not even know the name `UTC`; there the notebook uses the equivalent POSIX rule `UTC0` (zero offset, no daylight saving time). The saved outputs come from a local Windows run, so they show that name. The Olist timestamps themselves are local Brazilian time without an offset and are stored as `TIMESTAMP`, as the typed schema prescribes.
- `XDG_RUNTIME_DIR` is set only to silence a harmless `platformdirs` warning.

In [1]:
import glob, os, subprocess, sys, tempfile

os.environ.setdefault("XDG_RUNTIME_DIR", tempfile.mkdtemp())


def pip_install(*args):
    subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *args])


pip_install("jupysql", "psycopg2-binary", "sqlalchemy", "pandas", "kagglehub")

try:
    import pgserver
except ImportError:
    if sys.version_info >= (3, 13):
        tmp = tempfile.mkdtemp()
        subprocess.check_call([
            sys.executable, "-m", "pip", "download", "pgserver==0.1.4",
            "--no-deps", "--only-binary=:all:", "--python-version", "3.12",
            "-d", tmp, "-q",
        ])
        wheel = glob.glob(os.path.join(tmp, "pgserver-*.whl"))[0]
        fixed = wheel.replace("-cp312-cp312-", "-cp312-abi3-")
        os.rename(wheel, fixed)
        pip_install(fixed)
    else:
        pip_install("pgserver==0.1.4")
    import pgserver

print("Python", sys.version.split()[0], "| pgserver ready")

Python 3.11.0 | pgserver ready


In [2]:
from pathlib import Path

import pandas as pd
from sqlalchemy import create_engine, text
from sqlalchemy.exc import DBAPIError

PGDATA = Path(tempfile.gettempdir()) / "hw5_pg"
pg = pgserver.get_server(PGDATA, cleanup_mode="stop")
DB_URI = pg.get_uri().replace("postgresql://", "postgresql+psycopg2://", 1)

probe = create_engine(DB_URI, future=True)
with probe.connect() as conn:
    print(conn.execute(text("SELECT version()")).scalar_one())
    try:
        conn.execute(text("SET TIME ZONE 'UTC'"))
        SESSION_TZ = "UTC"
    except DBAPIError:
        SESSION_TZ = "UTC0"
probe.dispose()

engine = create_engine(DB_URI, future=True, connect_args={"options": f"-c timezone={SESSION_TZ}"})
print("session time zone:", SESSION_TZ)

PostgreSQL 16.2 on x86_64-pc-mingw64, compiled by gcc.exe (x86_64-posix-seh-rev2, Built by MinGW-W64 project) 12.2.0, 64-bit
session time zone: UTC0


In [3]:
def q(query: str) -> pd.DataFrame:
    with engine.connect() as conn:
        return pd.read_sql(text(query), conn)


%load_ext sql
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100
%sql engine

In [4]:
%%sql
SELECT current_setting('server_version') AS pg_version,
       current_setting('TimeZone')       AS session_tz,
       current_setting('work_mem')       AS work_mem,
       current_setting('jit')            AS jit,
       pg_jit_available()                AS jit_available;

pg_version,session_tz,work_mem,jit,jit_available
16.2,UTC0,4MB,on,False


### 1.2 Getting the data

**Source:** the Kaggle dataset [`olistbr/brazilian-ecommerce`](https://www.kaggle.com/datasets/olistbr/brazilian-ecommerce): about 100,000 orders placed on the Olist marketplace in Brazil between 2016 and 2018, split into related CSV files.

**Download.** `kagglehub` downloads public datasets without Kaggle credentials and caches them, so nothing has to be configured in Colab (variant A). If the CSV files are placed in `data/olist/` next to the notebook, for example from the course archive, the notebook uses that copy and downloads nothing (variant B).

**Stub schema.** The assignment also provides a small stub schema (30 customers, 50 orders, 100 items) for the case that Olist cannot be downloaded. It is not needed here, because the download works without credentials. The real data also makes the `EXPLAIN` plans in section 5 meaningful: on 100 rows, every plan is a handful of scans that finish in well under a millisecond.

Five of the nine files are used: customers, orders, order items, products and sellers.

In [5]:
import warnings

warnings.filterwarnings("ignore", message="IProgress not found")  # tqdm without ipywidgets

import kagglehub

files = {
    "olist_customers_raw": "olist_customers_dataset.csv",
    "olist_orders_raw": "olist_orders_dataset.csv",
    "olist_order_items_raw": "olist_order_items_dataset.csv",
    "olist_products_raw": "olist_products_dataset.csv",
    "olist_sellers_raw": "olist_sellers_dataset.csv",
}

LOCAL_DIR = Path.cwd() / "data" / "olist"
if all((LOCAL_DIR / file_name).exists() for file_name in files.values()):
    dataset_dir = LOCAL_DIR
    print("Source: local copy in data/olist/")
else:
    dataset_dir = Path(kagglehub.dataset_download("olistbr/brazilian-ecommerce"))
    print(f"Source: Kaggle olistbr/brazilian-ecommerce, version {dataset_dir.name}, via kagglehub")

for file_name in files.values():
    print(f"  {file_name:<34} {(dataset_dir / file_name).stat().st_size / 1e6:5.1f} MB")

Source: Kaggle olistbr/brazilian-ecommerce, version 2, via kagglehub
  olist_customers_dataset.csv          9.0 MB
  olist_orders_dataset.csv            17.7 MB
  olist_order_items_dataset.csv       15.4 MB
  olist_products_dataset.csv           2.4 MB
  olist_sellers_dataset.csv            0.2 MB


### 1.3 Raw tables

The raw tables mirror the CSV files one to one and serve only as a landing layer. `pandas` infers the column types and `to_sql(if_exists="replace")` recreates the tables on every run. All later sections read from the typed tables only.

In [6]:
for table_name, file_name in files.items():
    path = os.path.join(dataset_dir, file_name)
    df = pd.read_csv(path)
    df.to_sql(table_name, engine, if_exists="replace", index=False, chunksize=10_000)
    print(table_name, df.shape)

olist_customers_raw (99441, 5)


olist_orders_raw (99441, 8)


olist_order_items_raw (112650, 7)


olist_products_raw (32951, 9)
olist_sellers_raw (3095, 4)


### 1.4 Typed schema

The DDL is used exactly as given in the assignment; the cell only adds a `SELECT` at the end that lists the created tables. Olist's own IDs are the primary keys, an order item is identified by `(order_id, order_item_id)`, foreign keys link orders to customers and items to orders, products and sellers, and `CHECK` constraints cover the order status list and the non-negative sizes and prices.

The schema has no secondary indexes, only the indexes behind the primary keys. That matters in section 5: the pipeline has no selective filter, so every stage processes whole tables.

In [7]:
%%sql
DROP TABLE IF EXISTS olist_order_items CASCADE;
DROP TABLE IF EXISTS olist_orders      CASCADE;
DROP TABLE IF EXISTS olist_customers   CASCADE;
DROP TABLE IF EXISTS olist_products    CASCADE;
DROP TABLE IF EXISTS olist_sellers     CASCADE;

CREATE TABLE olist_customers (
    customer_id              TEXT PRIMARY KEY,
    customer_unique_id       TEXT NOT NULL,
    customer_zip_code_prefix INTEGER,
    customer_city            TEXT,
    customer_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_orders (
    order_id                      TEXT PRIMARY KEY,
    customer_id                   TEXT NOT NULL REFERENCES olist_customers(customer_id),
    order_status                  TEXT NOT NULL,
    order_purchase_timestamp      TIMESTAMP NOT NULL,
    order_approved_at             TIMESTAMP,
    order_delivered_carrier_date  TIMESTAMP,
    order_delivered_customer_date TIMESTAMP,
    order_estimated_delivery_date TIMESTAMP,
    CHECK (order_status IN (
        'created', 'approved', 'invoiced', 'processing',
        'shipped', 'delivered', 'unavailable', 'canceled'
    ))
);

CREATE TABLE olist_products (
    product_id                 TEXT PRIMARY KEY,
    product_category_name      TEXT,
    product_name_length        INTEGER CHECK (product_name_length IS NULL OR product_name_length >= 0),
    product_description_length INTEGER CHECK (product_description_length IS NULL OR product_description_length >= 0),
    product_photos_qty         INTEGER CHECK (product_photos_qty IS NULL OR product_photos_qty >= 0),
    product_weight_g           INTEGER CHECK (product_weight_g IS NULL OR product_weight_g >= 0),
    product_length_cm          INTEGER CHECK (product_length_cm IS NULL OR product_length_cm >= 0),
    product_height_cm          INTEGER CHECK (product_height_cm IS NULL OR product_height_cm >= 0),
    product_width_cm           INTEGER CHECK (product_width_cm IS NULL OR product_width_cm >= 0)
);

CREATE TABLE olist_sellers (
    seller_id              TEXT PRIMARY KEY,
    seller_zip_code_prefix INTEGER,
    seller_city            TEXT,
    seller_state           CHAR(2) NOT NULL
);

CREATE TABLE olist_order_items (
    order_id            TEXT NOT NULL REFERENCES olist_orders(order_id),
    order_item_id       INTEGER NOT NULL,
    product_id          TEXT REFERENCES olist_products(product_id),
    seller_id           TEXT REFERENCES olist_sellers(seller_id),
    shipping_limit_date TIMESTAMP,
    price               NUMERIC(12, 2) CHECK (price IS NULL OR price >= 0),
    freight_value       NUMERIC(12, 2) CHECK (freight_value IS NULL OR freight_value >= 0),
    PRIMARY KEY (order_id, order_item_id)
);

SELECT table_name, COUNT(*) AS n_columns
FROM information_schema.columns
WHERE table_schema = 'public'
  AND table_name LIKE 'olist%'
  AND table_name NOT LIKE '%raw'
GROUP BY table_name
ORDER BY table_name;

table_name,n_columns
olist_customers,5
olist_order_items,7
olist_orders,8
olist_products,9
olist_sellers,4


### 1.5 Moving the data into the typed tables

Each typed table is filled with `INSERT INTO ... SELECT` and an explicit cast per column, in foreign-key order. The source CSV spells two columns as `product_name_lenght` and `product_description_lenght`; the typed table uses the normalised names, while the `SELECT` reads the raw columns under their original names.

In [8]:
%%sql
INSERT INTO olist_customers
SELECT
    customer_id,
    customer_unique_id,
    customer_zip_code_prefix::INTEGER,
    customer_city,
    customer_state::CHAR(2)
FROM olist_customers_raw;

INSERT INTO olist_orders
SELECT
    order_id,
    customer_id,
    order_status,
    order_purchase_timestamp::TIMESTAMP,
    order_approved_at::TIMESTAMP,
    order_delivered_carrier_date::TIMESTAMP,
    order_delivered_customer_date::TIMESTAMP,
    order_estimated_delivery_date::TIMESTAMP
FROM olist_orders_raw;

INSERT INTO olist_products
SELECT
    product_id,
    product_category_name,
    product_name_lenght::INTEGER,
    product_description_lenght::INTEGER,
    product_photos_qty::INTEGER,
    product_weight_g::INTEGER,
    product_length_cm::INTEGER,
    product_height_cm::INTEGER,
    product_width_cm::INTEGER
FROM olist_products_raw;

INSERT INTO olist_sellers
SELECT
    seller_id,
    seller_zip_code_prefix::INTEGER,
    seller_city,
    seller_state::CHAR(2)
FROM olist_sellers_raw;

INSERT INTO olist_order_items
SELECT
    order_id,
    order_item_id::INTEGER,
    product_id,
    seller_id,
    shipping_limit_date::TIMESTAMP,
    price::NUMERIC(12, 2),
    freight_value::NUMERIC(12, 2)
FROM olist_order_items_raw;

ANALYZE olist_customers, olist_orders, olist_products, olist_sellers, olist_order_items;

SELECT relname                                     AS table_name,
       pg_size_pretty(pg_total_relation_size(oid)) AS size_with_indexes
FROM pg_class
WHERE relkind = 'r'
  AND relname LIKE 'olist%'
  AND relname NOT LIKE '%raw'
ORDER BY pg_total_relation_size(oid) DESC;

table_name,size_with_indexes
olist_order_items,24 MB
olist_orders,22 MB
olist_customers,19 MB
olist_products,6072 kB
olist_sellers,552 kB


`ANALYZE` after the load gives the planner fresh statistics right away instead of waiting for autovacuum, so the plans in section 5 do not depend on timing.

### 1.6 Smoke test of the row counts

In [9]:
%%sql
SELECT 'customers' AS table_name, COUNT(*) AS n_rows FROM olist_customers
UNION ALL SELECT 'orders', COUNT(*) FROM olist_orders
UNION ALL SELECT 'order_items', COUNT(*) FROM olist_order_items
UNION ALL SELECT 'products', COUNT(*) FROM olist_products
UNION ALL SELECT 'sellers', COUNT(*) FROM olist_sellers
ORDER BY table_name;

table_name,n_rows
customers,99441
order_items,112650
orders,99441
products,32951
sellers,3095


The same counts next to the raw tables:

In [10]:
%%sql
SELECT table_name, raw_rows, typed_rows, raw_rows - typed_rows AS lost_rows
FROM (
    SELECT 'customers' AS table_name,
           (SELECT COUNT(*) FROM olist_customers_raw) AS raw_rows,
           (SELECT COUNT(*) FROM olist_customers)     AS typed_rows
    UNION ALL
    SELECT 'orders',
           (SELECT COUNT(*) FROM olist_orders_raw),
           (SELECT COUNT(*) FROM olist_orders)
    UNION ALL
    SELECT 'order_items',
           (SELECT COUNT(*) FROM olist_order_items_raw),
           (SELECT COUNT(*) FROM olist_order_items)
    UNION ALL
    SELECT 'products',
           (SELECT COUNT(*) FROM olist_products_raw),
           (SELECT COUNT(*) FROM olist_products)
    UNION ALL
    SELECT 'sellers',
           (SELECT COUNT(*) FROM olist_sellers_raw),
           (SELECT COUNT(*) FROM olist_sellers)
) AS counts
ORDER BY table_name;

table_name,raw_rows,typed_rows,lost_rows
customers,99441,99441,0
order_items,112650,112650,0
orders,99441,99441,0
products,32951,32951,0
sellers,3095,3095,0


**Result:** every typed table has exactly as many rows as its raw table, so no row was lost and every row satisfied the keys, foreign keys and `CHECK` constraints. There is no shortfall to explain.

## 2. CTE-chain feature pipeline

The pipeline computes one row of features per `customer_id`. Every stage is a named CTE with one responsibility and a `-- Stage N:` comment, so each stage can be read, tested and timed on its own (section 5.4 does exactly that).

| Stage | CTE | Grain | Responsibility |
|---|---|---|---|
| 0 | `reference_date` | one row | Stable "today": the day after the last purchase in the data |
| 1 | `stage1_orders_total` | order | Order total (items plus freight), status, timestamp and date |
| 2 | `stage2_rfm` | customer | `recency_days`, `frequency`, `monetary`, first order date |
| 3 | `stage3_cohort` | customer | `cohort_quarter` from the first order date |
| 4 | `stage4_rolling_aov` | order | `avg_order_value_30d` over the 30 days up to each order, and the rank of the latest order |
| 5 | `stage5_cancel_rate` | customer | `cancellation_rate`: share of canceled orders |
| 6 | `stage6_category_revenue` | customer × category | Revenue per product category |
| 7 | `stage7_preferred_category` | customer | `preferred_category`: top-1 category by revenue |
| final | `SELECT` | customer | Joins the stages to `olist_customers` and fills gaps with explicit defaults |

A few details of the template:
- **Reference date.** `recency_days` is measured from `MAX(order_purchase_timestamp)::DATE + 1`, not from `CURRENT_DATE`, so the result does not change from day to day.
- **Rolling window.** `RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW` defines the window by time, not by a number of rows: it averages all orders of the customer from the 30 days up to and including the current order. The final `SELECT` takes the value at the most recent order (`rn_recent = 1`).
- **Defaults.** The final `LEFT JOIN`s keep every customer. A customer without orders would get `recency_days = 9999`, frequency 0 and monetary 0; a customer whose orders have no items gets the category `(none)`, and items of products without a category are grouped as `(unknown)`.
- **Tie-breaks.** `ROW_NUMBER()` with a second sort key (`order_id`, category name) makes the latest order and the top category deterministic when values are equal.

In [11]:
%%sql
WITH
reference_date AS (
    -- Stage 0: stable reference date for historical dataset
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    -- Stage 1: orders + order_total + status
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    -- Stage 2: RFM features per customer
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    -- Stage 3: cohort quarter from first order date
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    -- Stage 4: rolling 30-day average order value per customer
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    -- Stage 5: cancellation rate per customer
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    -- Stage 6: revenue by customer and product category
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    -- Stage 7: preferred category per customer via top-1 ranking
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id
ORDER BY monetary DESC NULLS LAST, c.customer_id
LIMIT 20;

customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0000,telefonia_fixa,13664.08
ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0000,telefonia_fixa,7274.88
c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0000,utilidades_domesticas,6929.31
f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0000,pcs,6922.21
3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0000,artes,6726.66
05455dfa7cd02f13d132aa7a6a9729c6,MG,328,1,6081.54,2017-11-24,2017-10-01,6081.54,0.0000,agro_industria_e_comercio,6081.54
df55c14d1476a9a3467f131269c2477f,RJ,565,1,4950.34,2017-04-01,2017-04-01,4950.34,0.0000,eletroportateis,4950.34
e0a2412720e9ea4f26c1ac985f6a7358,GO,98,1,4809.44,2018-07-12,2018-07-01,4809.44,1.0000,cool_stuff,4809.44
24bbf5fd2f2e1b359ee7de94defc4a15,SP,548,1,4764.34,2017-04-18,2017-04-01,4764.34,0.0000,eletroportateis,4764.34
3d979689f636322c62418b6346b1c6d2,PB,118,1,4681.78,2018-06-22,2018-04-01,4681.78,0.0000,pcs,4681.78


**Interpretation:** the top 20 customers by `monetary` all placed a single order: `frequency` is 1 in every row, and `avg_order_value_30d_recent` equals `monetary`, because a 30-day window over one order is that order. The leader `1617b135…` from RJ spent R$13,664.08 on fixed-line phones in September 2017, which gives a recency of 384 days at the reference date 2018-10-18. Two rows show limits of the template:
- `e0a24127…` (GO) has `cancellation_rate = 1.0000` but still `monetary = 4,809.44`. Stage 1 sums the items of every order regardless of its status, so canceled orders count as spend. For a model, `monetary` should count only orders that were not canceled, or the canceled value should become a separate feature.
- `3be2c536…` (PA) has the preferred category `(unknown)`: its product has no category, so the label from stage 6 wins.

### 2.1 Grain check: `customer_id` vs. `customer_unique_id`

The template keys every feature on `customer_id`. In Olist a new `customer_id` is issued for every order, so it is worth checking how many orders one key actually has:

In [12]:
%%sql
SELECT
    'customer_id' AS grain,
    COUNT(*)                                      AS n_keys,
    MAX(n_orders)                                 AS max_orders,
    COUNT(*) FILTER (WHERE n_orders > 1)          AS keys_with_repeat_orders
FROM (
    SELECT customer_id, COUNT(*) AS n_orders
    FROM olist_orders
    GROUP BY customer_id
) AS per_customer_id
UNION ALL
SELECT
    'customer_unique_id',
    COUNT(*),
    MAX(n_orders),
    COUNT(*) FILTER (WHERE n_orders > 1)
FROM (
    SELECT c.customer_unique_id, COUNT(*) AS n_orders
    FROM olist_orders AS o
    JOIN olist_customers AS c
        ON c.customer_id = o.customer_id
    GROUP BY c.customer_unique_id
) AS per_buyer;

grain,n_keys,max_orders,keys_with_repeat_orders
customer_id,99441,1,0
customer_unique_id,96096,17,2997


**Interpretation:** each of the 99,441 `customer_id` values has exactly one order, while the 96,096 real buyers (`customer_unique_id`) have up to 17 orders, and 2,997 of them ordered more than once. At the `customer_id` grain the pipeline cannot express repeat behaviour: `frequency` is always 1, `cancellation_rate` is either 0 or 1, the rolling AOV repeats the order total, and `recency_days` is just the age of that one order. The structure of the pipeline is right, but for a real customer model the same stages should be keyed on `customer_unique_id` by joining `olist_customers` in stages 1 and 6; only the key column changes, not the structure. This notebook keeps the template's key, so the view and the `EXPLAIN` comparison follow the assignment.

## 3. Recursive CTE: referral chain

### 3.1 Synthetic referral table

Olist has no referral data, so `hw5_referrals` is generated: the first ten customers by `customer_id` form one chain, where each customer referred the next one. The primary key on `referred_customer_id` means that a customer can be referred only once, which makes the referrals a forest of trees, and the `CHECK` rules out self-referrals. `ON CONFLICT DO NOTHING` makes the insert safe to repeat.

In [13]:
%%sql
DROP TABLE IF EXISTS hw5_referrals CASCADE;

CREATE TABLE hw5_referrals (
    referrer_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_customer_id TEXT NOT NULL REFERENCES olist_customers(customer_id),
    referred_at          TIMESTAMPTZ NOT NULL DEFAULT NOW(),
    PRIMARY KEY (referred_customer_id),
    CHECK (referrer_customer_id <> referred_customer_id)
);

WITH ordered_customers AS (
    SELECT
        customer_id,
        ROW_NUMBER() OVER (ORDER BY customer_id) AS rn
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 10
)
INSERT INTO hw5_referrals (referrer_customer_id, referred_customer_id)
SELECT
    prev.customer_id AS referrer_customer_id,
    nxt.customer_id AS referred_customer_id
FROM ordered_customers AS prev
JOIN ordered_customers AS nxt
    ON nxt.rn = prev.rn + 1
ON CONFLICT (referred_customer_id) DO NOTHING;

SELECT *
FROM hw5_referrals
ORDER BY referrer_customer_id, referred_customer_id;

referrer_customer_id,referred_customer_id,referred_at
00012a2ce6f8dcda20d059ce98491703,000161a058600d5901f007fab4c27140,2026-11-04 13:29:27.260158+00:00
000161a058600d5901f007fab4c27140,0001fd6190edaaf884bcaf3d49edf079,2026-11-04 13:29:27.260158+00:00
0001fd6190edaaf884bcaf3d49edf079,0002414f95344307404f0ace7a26f1d5,2026-11-04 13:29:27.260158+00:00
0002414f95344307404f0ace7a26f1d5,000379cdec625522490c315e70c7a9fb,2026-11-04 13:29:27.260158+00:00
000379cdec625522490c315e70c7a9fb,0004164d20a9e969af783496f3408652,2026-11-04 13:29:27.260158+00:00
0004164d20a9e969af783496f3408652,000419c5494106c306a97b5635748086,2026-11-04 13:29:27.260158+00:00
000419c5494106c306a97b5635748086,00046a560d407e99b969756e0b10f282,2026-11-04 13:29:27.260158+00:00
00046a560d407e99b969756e0b10f282,00050bf6e01e69d5c0fd612f1bcfb69c,2026-11-04 13:29:27.260158+00:00
00050bf6e01e69d5c0fd612f1bcfb69c,000598caf2ef4117407665ac33275130,2026-11-04 13:29:27.260158+00:00


### 3.2 Depth and path with `WITH RECURSIVE`

- **Anchor part.** Every customer who was not referred by anyone is a root at depth 1. `NOT EXISTS` expresses this anti-join correctly even if the subquery ever returned `NULL`; `NOT IN` would then return no rows at all, because `x NOT IN (..., NULL)` is never true.
- **Recursive part.** From each customer found so far, it follows the referral edges to the referred customers, adds 1 to the depth and appends the customer to the path.
- **Cycle protection.** `NOT r.referred_customer_id = ANY(rd.path_ids)` refuses to visit a customer that is already on the current path, and `rd.depth < 50` caps the depth even if the path check were missing or wrong.

In [14]:
%%sql
WITH RECURSIVE referral_depth AS (
    -- ANCHOR: root customers who were not referred by other customers
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids,
        c.customer_id::TEXT AS path_text
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS r
        WHERE r.referred_customer_id = c.customer_id
    )

    UNION ALL

    -- RECURSIVE: move from a referrer to the customers they referred
    SELECT
        r.referred_customer_id AS customer_id,
        rd.depth + 1 AS depth,
        rd.path_ids || r.referred_customer_id AS path_ids,
        rd.path_text || ' -> ' || r.referred_customer_id AS path_text
    FROM hw5_referrals AS r
    JOIN referral_depth AS rd
        ON rd.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(rd.path_ids)
      AND rd.depth < 50
)
SELECT
    depth,
    COUNT(*) AS n_customers,
    MAX(path_text) AS sample_path
FROM referral_depth
GROUP BY depth
ORDER BY depth;

depth,n_customers,sample_path
1,99432,ffffe8b65bbe3087b653a978c870db99
2,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140
3,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079
4,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5
5,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb
6,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652
7,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086
8,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282
9,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282 -> 00050bf6e01e69d5c0fd612f1bcfb69c
10,1,00012a2ce6f8dcda20d059ce98491703 -> 000161a058600d5901f007fab4c27140 -> 0001fd6190edaaf884bcaf3d49edf079 -> 0002414f95344307404f0ace7a26f1d5 -> 000379cdec625522490c315e70c7a9fb -> 0004164d20a9e969af783496f3408652 -> 000419c5494106c306a97b5635748086 -> 00046a560d407e99b969756e0b10f282 -> 00050bf6e01e69d5c0fd612f1bcfb69c -> 000598caf2ef4117407665ac33275130


**Interpretation:** the chain itself is found correctly: depths 2 to 10 hold one customer each, and the sample path at depth 10 lists all ten customers from `00012a2c…` to `000598ca…`. Depth 1, however, holds 99,432 customers, because the anchor treats every customer without a referrer as a root, and that is everyone except the nine referred customers. Only one of these roots ever referred anyone. The depth-1 row therefore describes the whole customer base rather than the referral program, and its `sample_path` is simply the largest `customer_id`. 99,432 + 9 = 99,441, so every customer appears exactly once.

### 3.3 The referral tree itself

For a referral report the anchor should start only from customers who actually referred someone. Here the roots are referrers who were not referred themselves (`SELECT DISTINCT`, because a root with several referrals would otherwise start several times). IDs are shortened to their first eight characters to keep the paths readable.

In [15]:
%%sql
WITH RECURSIVE referral_tree AS (
    -- ANCHOR: referrers who were not referred themselves
    SELECT DISTINCT
        r.referrer_customer_id AS customer_id,
        CAST(NULL AS TEXT) AS referred_by,
        1 AS depth,
        ARRAY[r.referrer_customer_id]::TEXT[] AS path_ids,
        LEFT(r.referrer_customer_id, 8) AS path_text
    FROM hw5_referrals AS r
    WHERE NOT EXISTS (
        SELECT 1
        FROM hw5_referrals AS up
        WHERE up.referred_customer_id = r.referrer_customer_id
    )

    UNION ALL

    -- RECURSIVE: one level down the tree
    SELECT
        r.referred_customer_id,
        r.referrer_customer_id,
        t.depth + 1,
        t.path_ids || r.referred_customer_id,
        t.path_text || ' -> ' || LEFT(r.referred_customer_id, 8)
    FROM hw5_referrals AS r
    JOIN referral_tree AS t
        ON t.customer_id = r.referrer_customer_id
    WHERE NOT r.referred_customer_id = ANY(t.path_ids)
      AND t.depth < 50
)
SELECT depth, LEFT(customer_id, 8) AS customer, LEFT(referred_by, 8) AS referred_by, path_text
FROM referral_tree
ORDER BY depth;

depth,customer,referred_by,path_text
1,00012a2c,None,00012a2c
2,000161a0,00012a2c,00012a2c -> 000161a0
3,0001fd61,000161a0,00012a2c -> 000161a0 -> 0001fd61
4,0002414f,0001fd61,00012a2c -> 000161a0 -> 0001fd61 -> 0002414f
5,000379cd,0002414f,00012a2c -> 000161a0 -> 0001fd61 -> 0002414f -> 000379cd
6,0004164d,000379cd,00012a2c -> 000161a0 -> 0001fd61 -> 0002414f -> 000379cd -> 0004164d
7,000419c5,0004164d,00012a2c -> 000161a0 -> 0001fd61 -> 0002414f -> 000379cd -> 0004164d -> 000419c5
8,00046a56,000419c5,00012a2c -> 000161a0 -> 0001fd61 -> 0002414f -> 000379cd -> 0004164d -> 000419c5 -> 00046a56
9,00050bf6,00046a56,00012a2c -> 000161a0 -> 0001fd61 -> 0002414f -> 000379cd -> 0004164d -> 000419c5 -> 00046a56 -> 00050bf6
10,000598ca,00050bf6,00012a2c -> 000161a0 -> 0001fd61 -> 0002414f -> 000379cd -> 0004164d -> 000419c5 -> 00046a56 -> 00050bf6 -> 000598ca


**Interpretation:** with the anchor restricted to real referrers, the result is exactly the referral tree: one root `00012a2c…` at depth 1 and nine referred customers below it, each with its referrer and the full path. This is the shape a referral report, or a `referral_depth` feature, would use.

### 3.4 What happens with a cycle A → B → A

The primary key allows a cycle as long as every customer is referred only once. To test it without changing `hw5_referrals`, the next queries read the edges from a CTE that adds one extra edge B → A, the reverse of the first referral A → B. A is the first customer of the chain.

First, the query from 3.2 on these edges:

In [16]:
%%sql
WITH RECURSIVE
edges AS (
    SELECT referrer_customer_id, referred_customer_id
    FROM hw5_referrals
    UNION ALL
    -- extra edge B -> A: together with A -> B it closes the cycle A -> B -> A
    SELECT referred_customer_id, referrer_customer_id
    FROM hw5_referrals
    WHERE referrer_customer_id = (SELECT MIN(customer_id) FROM olist_customers)
),
referral_depth AS (
    SELECT
        c.customer_id,
        1 AS depth,
        ARRAY[c.customer_id]::TEXT[] AS path_ids
    FROM olist_customers AS c
    WHERE NOT EXISTS (
        SELECT 1
        FROM edges AS e
        WHERE e.referred_customer_id = c.customer_id
    )

    UNION ALL

    SELECT
        e.referred_customer_id,
        rd.depth + 1,
        rd.path_ids || e.referred_customer_id
    FROM edges AS e
    JOIN referral_depth AS rd
        ON rd.customer_id = e.referrer_customer_id
    WHERE NOT e.referred_customer_id = ANY(rd.path_ids)
      AND rd.depth < 50
)
SELECT
    COUNT(*)                                                         AS rows_returned,
    MAX(depth)                                                       AS max_depth,
    (SELECT COUNT(*) FROM olist_customers)                           AS customers_total,
    (SELECT COUNT(*) FROM olist_customers) - COUNT(DISTINCT customer_id) AS customers_missing
FROM referral_depth;

rows_returned,max_depth,customers_total,customers_missing
99431,1,99441,10


**Interpretation:** with the extra edge, the result has 99,431 rows instead of 99,441, all at depth 1. The ten customers of the chain are missing, and there is no error. The reason is the anchor: after B → A, A is a referred customer too, so the chain has no root left that `NOT EXISTS` would accept, and the recursion never starts for it. A cycle without an entry from outside is invisible to a traversal that starts from roots, so a production pipeline should also check that every customer in the referral table is reachable from some root.

Second, a walk that starts directly at A, as a "whom did A bring in" query would. The same walk runs three times: with both guards from the assignment, with the depth guard only, and with PostgreSQL's `CYCLE` clause (available since version 14), which tracks the visited rows itself and marks the row that closes a cycle.

In [17]:
%%sql
WITH RECURSIVE
edges AS (
    SELECT referrer_customer_id, referred_customer_id
    FROM hw5_referrals
    UNION ALL
    SELECT referred_customer_id, referrer_customer_id
    FROM hw5_referrals
    WHERE referrer_customer_id = (SELECT MIN(customer_id) FROM olist_customers)
),
start_customer AS (
    SELECT MIN(customer_id) AS customer_id
    FROM olist_customers
),
path_and_depth_guard AS (
    SELECT s.customer_id, 1 AS depth, ARRAY[s.customer_id]::TEXT[] AS path_ids
    FROM start_customer AS s
    UNION ALL
    SELECT e.referred_customer_id, w.depth + 1, w.path_ids || e.referred_customer_id
    FROM edges AS e
    JOIN path_and_depth_guard AS w
        ON w.customer_id = e.referrer_customer_id
    WHERE NOT e.referred_customer_id = ANY(w.path_ids)
      AND w.depth < 50
),
depth_guard_only AS (
    SELECT s.customer_id, 1 AS depth
    FROM start_customer AS s
    UNION ALL
    SELECT e.referred_customer_id, w.depth + 1
    FROM edges AS e
    JOIN depth_guard_only AS w
        ON w.customer_id = e.referrer_customer_id
    WHERE w.depth < 50
),
cycle_clause AS (
    SELECT s.customer_id, 1 AS depth
    FROM start_customer AS s
    UNION ALL
    SELECT e.referred_customer_id, w.depth + 1
    FROM edges AS e
    JOIN cycle_clause AS w
        ON w.customer_id = e.referrer_customer_id
) CYCLE customer_id SET is_cycle USING visited
SELECT 'path_ids + depth guard' AS protection,
       COUNT(*) AS n_rows, COUNT(DISTINCT customer_id) AS distinct_customers,
       MAX(depth) AS max_depth, 0 AS rows_marked_as_cycle
FROM path_and_depth_guard
UNION ALL
SELECT 'depth guard only',
       COUNT(*), COUNT(DISTINCT customer_id), MAX(depth), 0
FROM depth_guard_only
UNION ALL
SELECT 'CYCLE clause',
       COUNT(*), COUNT(DISTINCT customer_id), MAX(depth),
       COUNT(*) FILTER (WHERE is_cycle)
FROM cycle_clause;

protection,n_rows,distinct_customers,max_depth,rows_marked_as_cycle
path_ids + depth guard,10,10,10,0
depth guard only,230,10,50,0
CYCLE clause,11,10,10,1


**Interpretation:**
- **Both guards.** The walk visits each of the ten customers once and stops. At B, the edge B → A is rejected because A is already in `path_ids`.
- **Depth guard only.** The walk runs A → B → A → B … and restarts the rest of the chain after every visit to B. It stops only because of `depth < 50`, and by then it has produced 230 rows for 10 distinct customers. Without the depth guard it would never end, because `UNION ALL` keeps adding the same customers at a greater depth.
- **`CYCLE` clause.** PostgreSQL keeps the visited `customer_id`s in the hidden `visited` array, emits the row that closes the cycle with `is_cycle = true` and does not follow it further. That gives 11 rows: the 10 customers plus one row marked as a cycle, which is useful for reporting broken data instead of hiding it.

## 4. Materialized view with concurrent refresh

### 4.1 Creating the view

`mv_olist_customer_features` stores the full result of the pipeline from section 2: all stages and all eleven feature columns, without the final `ORDER BY` and `LIMIT`. The unique index on `customer_id` serves two purposes: fast lookups by key, and it is the precondition for `REFRESH MATERIALIZED VIEW CONCURRENTLY`.

In [18]:
%%sql
DROP MATERIALIZED VIEW IF EXISTS mv_olist_customer_features CASCADE;

CREATE MATERIALIZED VIEW mv_olist_customer_features AS
WITH
reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
),
stage3_cohort AS (
    SELECT
        r.*,
        DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r
),
stage4_rolling_aov AS (
    SELECT
        s.customer_id,
        s.order_id,
        s.order_ts,
        s.order_total,
        AVG(s.order_total) OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts
            RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
        )::NUMERIC(12, 2) AS avg_order_value_30d,
        ROW_NUMBER() OVER (
            PARTITION BY s.customer_id
            ORDER BY s.order_ts DESC, s.order_id DESC
        ) AS rn_recent
    FROM stage1_orders_total AS s
),
stage5_cancel_rate AS (
    SELECT
        customer_id,
        AVG(
            CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END
        )::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id
),
stage6_category_revenue AS (
    SELECT
        o.customer_id,
        COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
        SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        )::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p
        ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')
),
stage7_preferred_category AS (
    SELECT
        customer_id,
        product_category_name,
        category_revenue
    FROM (
        SELECT
            cr.*,
            ROW_NUMBER() OVER (
                PARTITION BY cr.customer_id
                ORDER BY cr.category_revenue DESC, cr.product_category_name
            ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1
)
SELECT
    c.customer_id,
    c.customer_state,
    COALESCE(coh.recency_days, 9999) AS recency_days,
    COALESCE(coh.frequency, 0) AS frequency,
    COALESCE(coh.monetary, 0)::NUMERIC(12, 2) AS monetary,
    coh.first_order_date,
    coh.cohort_quarter,
    COALESCE(aov.avg_order_value_30d, 0)::NUMERIC(12, 2) AS avg_order_value_30d_recent,
    COALESCE(cancel.cancellation_rate, 0)::NUMERIC(6, 4) AS cancellation_rate,
    COALESCE(pc.product_category_name, '(none)') AS preferred_category,
    COALESCE(pc.category_revenue, 0)::NUMERIC(12, 2) AS preferred_category_revenue
FROM olist_customers AS c
LEFT JOIN stage3_cohort AS coh
    ON coh.customer_id = c.customer_id
LEFT JOIN stage4_rolling_aov AS aov
    ON aov.customer_id = c.customer_id
   AND aov.rn_recent = 1
LEFT JOIN stage5_cancel_rate AS cancel
    ON cancel.customer_id = c.customer_id
LEFT JOIN stage7_preferred_category AS pc
    ON pc.customer_id = c.customer_id;

CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id
    ON mv_olist_customer_features (customer_id);

SELECT indexname, indexdef
FROM pg_indexes
WHERE tablename = 'mv_olist_customer_features';

indexname,indexdef
idx_mv_olist_customer_features_customer_id,CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id ON public.mv_olist_customer_features USING btree (customer_id)


### 4.2 Checking the view

In [19]:
%%sql
SELECT COUNT(*) AS n_rows
FROM mv_olist_customer_features;

n_rows
99441


In [20]:
%%sql
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST, customer_id
LIMIT 5;

customer_id,customer_state,recency_days,frequency,monetary,first_order_date,cohort_quarter,avg_order_value_30d_recent,cancellation_rate,preferred_category,preferred_category_revenue
1617b1357756262bfa56ab541c47bc16,RJ,384,1,13664.08,2017-09-29,2017-07-01,13664.08,0.0000,telefonia_fixa,13664.08
ec5b2ba62e574342386871631fafd3fc,ES,95,1,7274.88,2018-07-15,2018-07-01,7274.88,0.0000,telefonia_fixa,7274.88
c6e2731c5b391845f6800c97401a43a9,MS,613,1,6929.31,2017-02-12,2017-01-01,6929.31,0.0000,utilidades_domesticas,6929.31
f48d464a0baaea338cb25f816991ab1f,ES,85,1,6922.21,2018-07-25,2018-07-01,6922.21,0.0000,pcs,6922.21
3fd6777bbce08a352fddd04e4a7cc8f6,SP,512,1,6726.66,2017-05-24,2017-04-01,6726.66,0.0000,artes,6726.66


The view makes it cheap to check every stage's output at once. Each column below tests one stage:

In [21]:
%%sql
SELECT
    COUNT(*)                                                         AS n_rows,
    COUNT(DISTINCT customer_id)                                      AS distinct_customers,
    COUNT(*) FILTER (WHERE recency_days = 9999)                      AS without_orders,
    COUNT(*) FILTER (WHERE frequency = 1)                            AS frequency_1,
    COUNT(*) FILTER (WHERE avg_order_value_30d_recent <> monetary)   AS rolling_aov_differs,
    COUNT(*) FILTER (WHERE cancellation_rate = 1)                    AS fully_canceled,
    COUNT(*) FILTER (WHERE cancellation_rate NOT IN (0, 1))          AS partly_canceled,
    COUNT(*) FILTER (WHERE preferred_category = '(none)')            AS no_items,
    COUNT(*) FILTER (WHERE preferred_category = '(unknown)')         AS unknown_category
FROM mv_olist_customer_features;

n_rows,distinct_customers,without_orders,frequency_1,rolling_aov_differs,fully_canceled,partly_canceled,no_items,unknown_category
99441,99441,0,99441,0,625,0,775,1423


In [22]:
%%sql
SELECT
    cohort_quarter,
    COUNT(*)                                AS n_customers,
    ROUND(AVG(monetary), 2)                 AS avg_monetary,
    ROUND(100 * AVG(cancellation_rate), 2)  AS cancel_pct,
    ROUND(AVG(recency_days))                AS avg_recency_days
FROM mv_olist_customer_features
GROUP BY cohort_quarter
ORDER BY cohort_quarter;

cohort_quarter,n_customers,avg_monetary,cancel_pct,avg_recency_days
2016-07-01,4,88.69,50.00,769
2016-10-01,325,174.86,7.38,741
2017-01-01,5262,162.58,1.01,599
2017-04-01,9349,160.61,0.67,517
2017-07-01,12642,156.11,0.59,428
2017-10-01,17848,157.55,0.41,334
2018-01-01,21208,153.21,0.63,245
2018-04-01,19979,166.78,0.29,157
2018-07-01,12820,160.86,1.09,79
2018-10-01,4,0.00,100.00,9


**Interpretation:** the view has one row per customer (99,441 rows, all distinct), and each stage behaves as section 2.1 predicts: `frequency` is 1 for every customer, the rolling AOV never differs from `monetary`, and the cancellation rate is 1 for exactly the 625 canceled orders and never fractional. 775 customers have no items and get `(none)`, and 1,423 get `(unknown)` as their preferred category.

The cohorts follow the growth of Olist, from 325 first orders in the fourth quarter of 2016 to 21,208 in the first quarter of 2018, with an average spend between R$153 and R$175 in every full quarter. The edges of the data show up as well: the third quarter of 2016 has only 4 customers, half of them canceled, and the fourth quarter of 2018 consists of 4 canceled orders without items. That data tail is what moves the reference date to 2018-10-18.

### 4.3 `REFRESH MATERIALIZED VIEW CONCURRENTLY`

A plain `REFRESH` rebuilds the view and holds an exclusive lock for the whole time, so every reader waits. `CONCURRENTLY` computes the new result next to the old one, compares the two through the unique index, and applies only the differences as inserts, updates and deletes. Readers keep seeing the old data until the refresh commits. The command runs in its own cell, after the unique index exists.

In [23]:
%%sql
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;

SELECT COUNT(*) AS n_rows_after_refresh
FROM mv_olist_customer_features;

n_rows_after_refresh
99441


**Why the unique index is required.** To match old and new rows, PostgreSQL needs a unique index on plain columns of the view, without a `WHERE` clause. A small demo view shows the error with an ordinary index and the success with a unique one:

In [24]:
demo_setup = """
DROP MATERIALIZED VIEW IF EXISTS hw5_mv_refresh_demo;
CREATE MATERIALIZED VIEW hw5_mv_refresh_demo AS
    SELECT customer_state, COUNT(*) AS n_customers
    FROM olist_customers
    GROUP BY customer_state;
CREATE INDEX hw5_mv_refresh_demo_state ON hw5_mv_refresh_demo (customer_state);
"""
with engine.begin() as conn:
    for statement in demo_setup.split(";"):
        if statement.strip():
            conn.execute(text(statement))

for attempt in ("non-unique index", "unique index"):
    if attempt == "unique index":
        with engine.begin() as conn:
            conn.execute(text("CREATE UNIQUE INDEX hw5_mv_refresh_demo_state_uq ON hw5_mv_refresh_demo (customer_state)"))
    try:
        with engine.begin() as conn:
            conn.execute(text("REFRESH MATERIALIZED VIEW CONCURRENTLY hw5_mv_refresh_demo"))
        print(f"{attempt}: refresh succeeded")
    except DBAPIError as error:
        message = str(error.orig).strip().splitlines()
        print(f"{attempt}: {message[0]}")
        print(" " * (len(attempt) + 2) + message[-1])

with engine.begin() as conn:
    conn.execute(text("DROP MATERIALIZED VIEW hw5_mv_refresh_demo"))

non-unique index: cannot refresh materialized view "public.hw5_mv_refresh_demo" concurrently
                  HINT:  Create a unique index with no WHERE clause on one or more columns of the materialized view.
unique index: refresh succeeded


### 4.4 Freshness: the view lags behind the tables until it is refreshed

A view is a snapshot. To see that, the order of the customer with the highest `monetary` is set to `canceled`, as if the cancellation arrived after the last refresh:

In [25]:
%%sql
UPDATE olist_orders
SET order_status = 'canceled'
WHERE customer_id = (
    SELECT customer_id
    FROM mv_olist_customer_features
    ORDER BY monetary DESC, customer_id
    LIMIT 1
)
  AND order_status = 'delivered'
RETURNING order_id, customer_id, order_status;

order_id,customer_id,order_status
03caa2c082116e1d31e67e9ae3700499,1617b1357756262bfa56ab541c47bc16,canceled


The view still shows the old value, while the same feature computed from the tables already shows the cancellation:

In [26]:
%%sql
SELECT
    mv.customer_id,
    mv.cancellation_rate AS mv_cancellation_rate,
    (
        SELECT AVG(CASE WHEN o.order_status = 'canceled' THEN 1.0 ELSE 0.0 END)::NUMERIC(6, 4)
        FROM olist_orders AS o
        WHERE o.customer_id = mv.customer_id
    ) AS live_cancellation_rate
FROM mv_olist_customer_features AS mv
WHERE mv.customer_id = (
    SELECT customer_id
    FROM mv_olist_customer_features
    ORDER BY monetary DESC, customer_id
    LIMIT 1
);

customer_id,mv_cancellation_rate,live_cancellation_rate
1617b1357756262bfa56ab541c47bc16,0.0000,1.0000


After a concurrent refresh the view catches up:

In [27]:
%%sql
REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;

SELECT
    mv.customer_id,
    mv.cancellation_rate AS mv_cancellation_rate,
    (
        SELECT AVG(CASE WHEN o.order_status = 'canceled' THEN 1.0 ELSE 0.0 END)::NUMERIC(6, 4)
        FROM olist_orders AS o
        WHERE o.customer_id = mv.customer_id
    ) AS live_cancellation_rate
FROM mv_olist_customer_features AS mv
WHERE mv.customer_id = (
    SELECT customer_id
    FROM mv_olist_customer_features
    ORDER BY monetary DESC, customer_id
    LIMIT 1
);

customer_id,mv_cancellation_rate,live_cancellation_rate
1617b1357756262bfa56ab541c47bc16,1.0000,1.0000


The order is set back to `delivered` and the view is refreshed again, so the following sections work with the original data:

In [28]:
%%sql
UPDATE olist_orders
SET order_status = 'delivered'
WHERE customer_id = (
    SELECT customer_id
    FROM mv_olist_customer_features
    ORDER BY monetary DESC, customer_id
    LIMIT 1
)
  AND order_status = 'canceled';

REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features;

SELECT
    mv.customer_id,
    mv.cancellation_rate AS mv_cancellation_rate,
    (
        SELECT AVG(CASE WHEN o.order_status = 'canceled' THEN 1.0 ELSE 0.0 END)::NUMERIC(6, 4)
        FROM olist_orders AS o
        WHERE o.customer_id = mv.customer_id
    ) AS live_cancellation_rate
FROM mv_olist_customer_features AS mv
WHERE mv.customer_id = (
    SELECT customer_id
    FROM mv_olist_customer_features
    ORDER BY monetary DESC, customer_id
    LIMIT 1
);

customer_id,mv_cancellation_rate,live_cancellation_rate
1617b1357756262bfa56ab541c47bc16,0.0000,0.0000


### 4.5 Cost of a refresh

Both refresh variants timed three times each, keeping the fastest run:

In [29]:
import time


def best_time(statement: str, repeat: int = 3) -> float:
    timings = []
    for _ in range(repeat):
        start = time.perf_counter()
        with engine.begin() as conn:
            conn.execute(text(statement))
        timings.append(time.perf_counter() - start)
    return min(timings)


refresh_times = {
    "REFRESH": best_time("REFRESH MATERIALIZED VIEW mv_olist_customer_features"),
    "REFRESH CONCURRENTLY": best_time("REFRESH MATERIALIZED VIEW CONCURRENTLY mv_olist_customer_features"),
}
for name, seconds in refresh_times.items():
    print(f"{name:<22} {seconds * 1000:8.0f} ms")
print(f"CONCURRENTLY / plain: {refresh_times['REFRESH CONCURRENTLY'] / refresh_times['REFRESH']:.1f}x")

REFRESH                    2844 ms
REFRESH CONCURRENTLY       3763 ms
CONCURRENTLY / plain: 1.3x


**Interpretation:** a refresh recomputes the whole pipeline, so it costs about as much as the pipeline itself (5.4) plus writing 99,441 rows and the index: several seconds on this machine. `CONCURRENTLY` is slower than the plain refresh, because it builds the new result in a temporary table and then compares it row by row with the old one through the unique index. That is the price for not blocking readers. When little has changed, as here, the comparison writes almost nothing back, but it still has to compute and compare everything.

## 5. `EXPLAIN (ANALYZE, BUFFERS)`

`EXPLAIN ANALYZE` returns the plan as rows of text. In a table cell the indentation that shows the plan tree would be lost, so a small helper runs each statement and prints the plan as it is. It also keeps the timings and buffer counts for the summary in 5.5.

Timings depend on the machine and vary slightly between runs; the plan shapes and buffer counts do not. The settings in 1.1 show `jit = on`, but `pg_jit_available()` is false in this `pgserver` build, so the plans have no JIT section. On a build with LLVM support, the full pipeline, whose cost is above `jit_above_cost`, would also show a `JIT` block, and its compile time would add to the execution time.

In [30]:
import re

explain_runs = {}


def explain(label: str, statement: str) -> None:
    """Run an EXPLAIN (ANALYZE, BUFFERS) statement and print the plan with its indentation."""
    with engine.connect() as conn:
        lines = conn.execute(text(statement)).scalars().all()
    plan = "\n".join(lines)
    print(plan)
    root_buffers = re.search(r"Buffers: shared(?: hit=(\d+))?(?: read=(\d+))?", plan)
    explain_runs[label] = {
        "planning_ms": float(re.search(r"Planning Time: ([\d.]+) ms", plan).group(1)),
        "execution_ms": float(re.search(r"Execution Time: ([\d.]+) ms", plan).group(1)),
        "shared_hit": int(root_buffers.group(1) or 0) if root_buffers else 0,
        "shared_read": int(root_buffers.group(2) or 0) if root_buffers else 0,
        "seq_scans": len(re.findall(r"Seq Scan", plan)),
        "index_scans": len(re.findall(r"Index (?:Only )?Scan", plan)),
    }

### 5.1 Recomputing the pipeline: RFM stages 0–2

In [31]:
explain("pipeline: RFM stages 0-2", """
EXPLAIN (ANALYZE, BUFFERS)
WITH
reference_date AS (
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders
),
stage1_orders_total AS (
    SELECT
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp AS order_ts,
        o.order_purchase_timestamp::DATE AS order_date,
        COALESCE(SUM(
            COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)
        ), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi
        ON oi.order_id = o.order_id
    GROUP BY
        o.customer_id,
        o.order_id,
        o.order_status,
        o.order_purchase_timestamp
),
stage2_rfm AS (
    SELECT
        s.customer_id,
        (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
        COUNT(*) AS frequency,
        SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
        MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date
)
SELECT *
FROM stage2_rfm
ORDER BY monetary DESC NULLS LAST
LIMIT 10
""")

Limit  (cost=33620.68..33620.70 rows=10 width=65) (actual time=389.923..389.928 rows=10 loops=1)
  Buffers: shared hit=105262, temp read=958 written=1694
  ->  Sort  (cost=33620.68..33621.18 rows=200 width=65) (actual time=389.922..389.925 rows=10 loops=1)
        Sort Key: stage2_rfm.monetary DESC NULLS LAST
        Sort Method: top-N heapsort  Memory: 27kB
        Buffers: shared hit=105262, temp read=958 written=1694
        ->  Subquery Scan on stage2_rfm  (cost=33610.85..33616.35 rows=200 width=65) (actual time=302.140..374.149 rows=99441 loops=1)
              Buffers: shared hit=105262, temp read=958 written=1694
              ->  HashAggregate  (cost=33610.85..33614.35 rows=200 width=69) (actual time=302.138..367.174 rows=99441 loops=1)
                    Group Key: o.customer_id, (((max(olist_orders.order_purchase_timestamp))::date + 1))
                    Batches: 21  Memory Usage: 8265kB  Disk Usage: 7400kB
                    Buffers: shared hit=105262, temp read=958 writ

**Interpretation:** the RFM part alone takes about a second. Read from the bottom up:
- **Stage 1** is a `Merge Left Join` of two `Index Scan`s on the primary keys of orders and items, followed by a `GroupAggregate` per `order_id`. The planner walks both tables in `order_id` order to avoid a sort, but that order is unrelated to the physical order of the rows, so the scan of `olist_orders` touches a buffer for almost every row: 100,347 buffer hits for 99,441 rows, while a sequential scan of the same table reads 1,809 pages. This step takes most of the time.
- **Stage 0** is the one-row `Aggregate` over a `Seq Scan` of the orders, attached to stage 1 by a `Nested Loop`. It is cheap, about 20 ms.
- **Stage 2** is a `HashAggregate` over 99,441 customers. Its hash table does not fit into `work_mem` (4 MB), so it spills to disk in 21 batches (`Disk Usage: 7400kB`).
- The final `ORDER BY ... LIMIT 10` is a `top-N heapsort` that keeps only 10 rows in memory, so sorting the result costs almost nothing.

The CTEs do not appear in the plan: each one is referenced only once here, so PostgreSQL inlines it into the main query.

### 5.2 Lookup of one customer in the view

In [32]:
explain("view: lookup by customer_id", """
EXPLAIN (ANALYZE, BUFFERS)
WITH target_customer AS (
    SELECT customer_id
    FROM olist_customers
    ORDER BY customer_id
    LIMIT 1
)
SELECT mv.*
FROM mv_olist_customer_features AS mv
JOIN target_customer AS t
    ON t.customer_id = mv.customer_id
""")

Nested Loop  (cost=0.83..8.96 rows=1 width=158) (actual time=0.138..0.139 rows=1 loops=1)
  Buffers: shared hit=5 read=3
  ->  Limit  (cost=0.42..0.53 rows=1 width=33) (actual time=0.076..0.077 rows=1 loops=1)
        Buffers: shared hit=4
        ->  Index Only Scan using olist_customers_pkey on olist_customers  (cost=0.42..11128.03 rows=99441 width=33) (actual time=0.075..0.075 rows=1 loops=1)
              Heap Fetches: 1
              Buffers: shared hit=4
  ->  Index Scan using idx_mv_olist_customer_features_customer_id on mv_olist_customer_features mv  (cost=0.42..8.44 rows=1 width=158) (actual time=0.057..0.057 rows=1 loops=1)
        Index Cond: (customer_id = olist_customers.customer_id)
        Buffers: shared hit=1 read=3
Planning:
  Buffers: shared hit=7
Planning Time: 1.074 ms
Execution Time: 0.156 ms


**Interpretation:** the lookup is a `Nested Loop` of two index accesses. An `Index Only Scan` on `olist_customers_pkey` finds the first `customer_id` and stops after one row thanks to the `Limit`, and an `Index Scan` on the unique index of the view fetches the matching row. The query touches 8 buffers and finishes in a fraction of a millisecond, against about a second for the RFM part alone.

### 5.3 Top-N from the view, without and with an index on `monetary`

Without an index on `monetary`, the view has to be read completely:

In [33]:
explain("view: top-10 by monetary, no index", """
EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10
""")

Limit  (cost=4449.01..4450.16 rows=10 width=158) (actual time=26.481..29.753 rows=10 loops=1)
  Buffers: shared hit=1636
  ->  Gather Merge  (cost=4449.01..11175.94 rows=58495 width=158) (actual time=26.479..29.748 rows=10 loops=1)
        Workers Planned: 1
        Workers Launched: 1
        Buffers: shared hit=1636
        ->  Sort  (cost=3449.00..3595.24 rows=58495 width=158) (actual time=7.966..7.967 rows=5 loops=2)
              Sort Key: monetary DESC NULLS LAST
              Sort Method: top-N heapsort  Memory: 26kB
              Buffers: shared hit=1636
              Worker 0:  Sort Method: quicksort  Memory: 25kB
              ->  Parallel Seq Scan on mv_olist_customer_features  (cost=0.00..2184.95 rows=58495 width=158) (actual time=0.004..3.060 rows=49721 loops=2)
                    Buffers: shared hit=1600
Planning Time: 0.071 ms
Execution Time: 29.776 ms


With the index from the assignment:

In [34]:
%%sql
CREATE INDEX IF NOT EXISTS idx_mv_olist_customer_features_monetary
    ON mv_olist_customer_features (monetary DESC);

SELECT indexname, indexdef
FROM pg_indexes
WHERE tablename = 'mv_olist_customer_features'
ORDER BY indexname;

indexname,indexdef
idx_mv_olist_customer_features_customer_id,CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id ON public.mv_olist_customer_features USING btree (customer_id)
idx_mv_olist_customer_features_monetary,CREATE INDEX idx_mv_olist_customer_features_monetary ON public.mv_olist_customer_features USING btree (monetary DESC)


In [35]:
explain("view: top-10 by monetary, index (monetary DESC)", """
EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10
""")

Limit  (cost=4449.01..4450.16 rows=10 width=158) (actual time=27.606..30.468 rows=10 loops=1)
  Buffers: shared hit=1636
  ->  Gather Merge  (cost=4449.01..11175.94 rows=58495 width=158) (actual time=27.604..30.463 rows=10 loops=1)
        Workers Planned: 1
        Workers Launched: 1
        Buffers: shared hit=1636
        ->  Sort  (cost=3449.00..3595.24 rows=58495 width=158) (actual time=7.887..7.888 rows=5 loops=2)
              Sort Key: monetary DESC NULLS LAST
              Sort Method: top-N heapsort  Memory: 26kB
              Buffers: shared hit=1636
              Worker 0:  Sort Method: quicksort  Memory: 25kB
              ->  Parallel Seq Scan on mv_olist_customer_features  (cost=0.00..2184.95 rows=58495 width=158) (actual time=0.004..3.034 rows=49721 loops=2)
                    Buffers: shared hit=1600
Planning:
  Buffers: shared hit=20 read=1
Planning Time: 1.609 ms
Execution Time: 30.491 ms


**Interpretation:** the plan did not change. It is still a `Parallel Seq Scan` over the whole view with a `top-N heapsort`, 1,636 buffers and tens of milliseconds: the index exists but is not used. The reason is the `NULL` ordering. An index on `monetary DESC` stores `NULL`s first, because `DESC` implies `NULLS FIRST`, while the query asks for `NULLS LAST`. The planner cannot know that `monetary` never is `NULL`, because a materialized view carries no `NOT NULL` constraints, so the index order does not match the requested order. The index has to match the query exactly:

In [36]:
%%sql
DROP INDEX IF EXISTS idx_mv_olist_customer_features_monetary;

CREATE INDEX idx_mv_olist_customer_features_monetary
    ON mv_olist_customer_features (monetary DESC NULLS LAST);

SELECT indexname, indexdef
FROM pg_indexes
WHERE tablename = 'mv_olist_customer_features'
ORDER BY indexname;

indexname,indexdef
idx_mv_olist_customer_features_customer_id,CREATE UNIQUE INDEX idx_mv_olist_customer_features_customer_id ON public.mv_olist_customer_features USING btree (customer_id)
idx_mv_olist_customer_features_monetary,CREATE INDEX idx_mv_olist_customer_features_monetary ON public.mv_olist_customer_features USING btree (monetary DESC NULLS LAST)


In [37]:
explain("view: top-10 by monetary, index (monetary DESC NULLS LAST)", """
EXPLAIN (ANALYZE, BUFFERS)
SELECT *
FROM mv_olist_customer_features
ORDER BY monetary DESC NULLS LAST
LIMIT 10
""")

Limit  (cost=0.42..1.32 rows=10 width=158) (actual time=0.025..0.035 rows=10 loops=1)
  Buffers: shared hit=10 read=3
  ->  Index Scan using idx_mv_olist_customer_features_monetary on mv_olist_customer_features  (cost=0.42..8996.03 rows=99441 width=158) (actual time=0.024..0.032 rows=10 loops=1)
        Buffers: shared hit=10 read=3
Planning:
  Buffers: shared hit=15 read=1
Planning Time: 1.534 ms
Execution Time: 0.044 ms


**Interpretation:** with `monetary DESC NULLS LAST`, the plan becomes a `Limit` over an `Index Scan`. The index already returns the rows in the requested order, so PostgreSQL reads the first ten entries and stops: 13 buffers instead of 1,636, and about a tenth of a millisecond. The alternative would be to drop `NULLS LAST` from the query, which is safe here because `COALESCE` rules out `NULL`s.

### 5.4 Full recomputation and the cost of each feature

The plan in 5.1 covers only the RFM stages. The full pipeline is exactly what every refresh of the view recomputes, and PostgreSQL keeps that query: `pg_get_viewdef` returns the definition of the view, so it can be explained without copying it a fourth time.

In [38]:
with engine.connect() as conn:
    full_pipeline = conn.execute(
        text("SELECT pg_get_viewdef('mv_olist_customer_features'::regclass)")
    ).scalar_one().rstrip().rstrip(";")

explain("pipeline: all stages (view definition)", "EXPLAIN (ANALYZE, BUFFERS)\n" + full_pipeline)

Hash Left Join  (cost=86402.48..112991.11 rows=99441 width=150) (actual time=2014.129..3998.540 rows=99441 loops=1)
  Hash Cond: (c.customer_id = cancel.customer_id)
  Buffers: shared hit=111713, temp read=8432 written=8582
  CTE stage1_orders_total
    ->  GroupAggregate  (cost=0.83..27075.78 rows=99496 width=103) (actual time=0.045..243.404 rows=99441 loops=1)
          Group Key: o_1.order_id
          Buffers: shared hit=103453
          ->  Merge Left Join  (cost=0.83..24489.72 rows=112650 width=95) (actual time=0.023..162.488 rows=113425 loops=1)
                Merge Cond: (o_1.order_id = oi_1.order_id)
                Buffers: shared hit=103453
                ->  Index Scan using olist_orders_pkey on olist_orders o_1  (cost=0.42..12568.57 rows=99496 width=83) (actual time=0.009..53.292 rows=99441 loops=1)
                      Buffers: shared hit=100347
                ->  Index Scan using olist_order_items_pkey on olist_order_items oi_1  (cost=0.42..10264.29 rows=112650 width

To see which feature is the most expensive, each feature is computed on its own: the same stage definitions as in section 2, but only the stages that feature needs. Each query runs three times under `EXPLAIN (ANALYZE, FORMAT JSON)`, and the table shows the median execution time, the rows the last stage returns, and the main operations in its plan.

In [39]:
import json
import statistics

STAGES = {
    "reference_date": """
    SELECT (MAX(order_purchase_timestamp)::DATE + 1) AS as_of_date
    FROM olist_orders""",
    "stage1_orders_total": """
    SELECT o.customer_id, o.order_id, o.order_status,
           o.order_purchase_timestamp AS order_ts,
           o.order_purchase_timestamp::DATE AS order_date,
           COALESCE(SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0)), 0)::NUMERIC(12, 2) AS order_total
    FROM olist_orders AS o
    LEFT JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    GROUP BY o.customer_id, o.order_id, o.order_status, o.order_purchase_timestamp""",
    "stage2_rfm": """
    SELECT s.customer_id,
           (rd.as_of_date - MAX(s.order_date))::INTEGER AS recency_days,
           COUNT(*) AS frequency,
           SUM(s.order_total)::NUMERIC(12, 2) AS monetary,
           MIN(s.order_date) AS first_order_date
    FROM stage1_orders_total AS s
    CROSS JOIN reference_date AS rd
    GROUP BY s.customer_id, rd.as_of_date""",
    "stage3_cohort": """
    SELECT r.*, DATE_TRUNC('quarter', r.first_order_date)::DATE AS cohort_quarter
    FROM stage2_rfm AS r""",
    "stage4_rolling_aov": """
    SELECT s.customer_id, s.order_id, s.order_ts, s.order_total,
           AVG(s.order_total) OVER (
               PARTITION BY s.customer_id ORDER BY s.order_ts
               RANGE BETWEEN INTERVAL '30 days' PRECEDING AND CURRENT ROW
           )::NUMERIC(12, 2) AS avg_order_value_30d,
           ROW_NUMBER() OVER (
               PARTITION BY s.customer_id ORDER BY s.order_ts DESC, s.order_id DESC
           ) AS rn_recent
    FROM stage1_orders_total AS s""",
    "stage5_cancel_rate": """
    SELECT customer_id,
           AVG(CASE WHEN order_status = 'canceled' THEN 1.0 ELSE 0.0 END)::NUMERIC(6, 4) AS cancellation_rate
    FROM stage1_orders_total
    GROUP BY customer_id""",
    "stage6_category_revenue": """
    SELECT o.customer_id,
           COALESCE(p.product_category_name, '(unknown)') AS product_category_name,
           SUM(COALESCE(oi.price, 0) + COALESCE(oi.freight_value, 0))::NUMERIC(12, 2) AS category_revenue
    FROM olist_orders AS o
    JOIN olist_order_items AS oi ON oi.order_id = o.order_id
    LEFT JOIN olist_products AS p ON p.product_id = oi.product_id
    GROUP BY o.customer_id, COALESCE(p.product_category_name, '(unknown)')""",
    "stage7_preferred_category": """
    SELECT customer_id, product_category_name, category_revenue
    FROM (
        SELECT cr.*, ROW_NUMBER() OVER (
                   PARTITION BY cr.customer_id
                   ORDER BY cr.category_revenue DESC, cr.product_category_name
               ) AS rn
        FROM stage6_category_revenue AS cr
    ) AS ranked
    WHERE rn = 1""",
}

FEATURES = {
    "order totals (stage 1)": (
        ["stage1_orders_total"], "SELECT * FROM stage1_orders_total"),
    "RFM + cohort (stages 0-3)": (
        ["reference_date", "stage1_orders_total", "stage2_rfm", "stage3_cohort"],
        "SELECT * FROM stage3_cohort"),
    "rolling 30-day AOV (stages 1, 4)": (
        ["stage1_orders_total", "stage4_rolling_aov"],
        "SELECT * FROM stage4_rolling_aov WHERE rn_recent = 1"),
    "cancellation rate (stages 1, 5)": (
        ["stage1_orders_total", "stage5_cancel_rate"], "SELECT * FROM stage5_cancel_rate"),
    "preferred category (stages 6-7)": (
        ["stage6_category_revenue", "stage7_preferred_category"],
        "SELECT * FROM stage7_preferred_category"),
}


def build_query(stage_names, final_select):
    ctes = ",\n".join(f"{name} AS ({STAGES[name]}\n)" for name in stage_names)
    return f"WITH\n{ctes}\n{final_select}"


def plan_json(query):
    with engine.connect() as conn:
        result = conn.execute(text("EXPLAIN (ANALYZE, FORMAT JSON) " + query)).scalar_one()
    return (json.loads(result) if isinstance(result, str) else result)[0]


def main_operations(node, found=None):
    found = [] if found is None else found
    kind = node["Node Type"]
    if kind == "Aggregate":
        kind = {"Hashed": "HashAggregate", "Sorted": "GroupAggregate"}.get(node.get("Strategy"), "Aggregate")
    elif kind in ("Seq Scan", "Index Scan", "Index Only Scan"):
        kind = f"{kind} on {node['Relation Name']}"
    elif kind == "Sort" and "Sort Method" in node:
        kind = f"Sort ({node['Sort Method']})"
    skip = {"Result", "Subquery Scan", "CTE Scan", "Hash", "Materialize", "Limit", "Gather", "Gather Merge"}
    if kind not in skip and kind not in found:
        found.append(kind)
    for child in node.get("Plans", []):
        main_operations(child, found)
    return found


feature_costs = []
for feature, (stage_names, final_select) in FEATURES.items():
    query = build_query(stage_names, final_select)
    runs = [plan_json(query) for _ in range(3)]
    feature_costs.append({
        "feature": feature,
        "rows": runs[-1]["Plan"]["Actual Rows"],
        "median_ms": round(statistics.median(run["Execution Time"] for run in runs), 1),
        "main operations": ", ".join(main_operations(runs[-1]["Plan"])),
    })

pd.set_option("display.max_colwidth", None)
pd.DataFrame(feature_costs)

,feature,rows,median_ms,main operations
0,order totals (stage 1),99441,262.0,"GroupAggregate, Merge Join, Index Scan on olist_orders, Index Scan on olist_order_items"
1,RFM + cohort (stages 0-3),99441,498.5,"HashAggregate, Nested Loop, Aggregate, Seq Scan on olist_orders, GroupAggregate, Merge Join, Index Scan on olist_orders, Index Scan on olist_order_items"
2,"rolling 30-day AOV (stages 1, 4)",99441,1296.7,"WindowAgg, Incremental Sort, Sort (external merge), GroupAggregate, Merge Join, Index Scan on olist_orders, Index Scan on olist_order_items"
3,"cancellation rate (stages 1, 5)",99441,862.8,"HashAggregate, GroupAggregate, Sort (external merge), Hash Join, Seq Scan on olist_order_items, Seq Scan on olist_orders"
4,preferred category (stages 6-7),98666,754.8,"WindowAgg, Incremental Sort, GroupAggregate, Sort (external merge), Hash Join, Seq Scan on olist_order_items, Seq Scan on olist_orders, Seq Scan on olist_products"


**Interpretation:**
- **Full recomputation.** The whole pipeline takes several seconds, about four to five times as long as the RFM part. Stage 1 now appears as `CTE stage1_orders_total`, computed once and read by three `CTE Scan`s, because stages 2, 4 and 5 all use it, and PostgreSQL materializes a CTE that is referenced more than once. The plan reads the base tables with 5 `Seq Scan`s and 2 `Index Scan`s and writes thousands of temporary blocks, because several sorts and hash aggregates exceed `work_mem`.
- **Most expensive feature: the rolling 30-day AOV.** On its own it takes roughly three times as long as stage 1 alone, so stage 4 adds more than a second. Its two window functions need two different orders (`order_ts` for the rolling average, `order_ts DESC, order_id DESC` for `ROW_NUMBER`). The plan sorts all 99,441 orders (`Sort Method: external merge  Disk: 9440kB`), runs one `WindowAgg`, re-sorts with an `Incremental Sort` and runs the second `WindowAgg`. The cost comes from `Sort` and the window functions, not from a join.
- **Preferred category** costs about as much as RFM: a `Hash Join` of items, orders and products, an `external merge` sort for the `GroupAggregate` per customer and category, and another `WindowAgg` for the top-1 ranking.
- **RFM, cohort and cancellation rate** add little on top of stage 1: each is a `HashAggregate` per customer, which spills in 21 batches but stays fast.
- **Estimates.** The planner has no statistics for aggregated CTE output and assumes 200 rows, while 99,441 arrive. The hash tables built on those rows have to grow at run time (`Buckets: 131072 (originally 1024)`, `Batches: 2 (originally 1)`).

### 5.5 Summary of the `EXPLAIN` runs

In [40]:
summary = pd.DataFrame.from_dict(explain_runs, orient="index")
summary.index.name = "scenario"
summary

,planning_ms,execution_ms,shared_hit,shared_read,seq_scans,index_scans
scenario,,,,,,
pipeline: RFM stages 0-2,0.350,406.316,105262,0,1,2
view: lookup by customer_id,1.074,0.156,5,3,0,2
"view: top-10 by monetary, no index",0.071,29.776,1636,0,1,0
"view: top-10 by monetary, index (monetary DESC)",1.609,30.491,1636,0,1,0
"view: top-10 by monetary, index (monetary DESC NULLS LAST)",1.534,0.044,10,3,0,1
pipeline: all stages (view definition),0.985,4097.452,111713,0,5,2


**Interpretation:** reading features from the view is three to four orders of magnitude cheaper than computing them. Looking up one customer takes a fraction of a millisecond and 8 buffers, while the full recomputation takes seconds, over 110,000 buffer hits and temporary files. Even reading the whole view, as the top-10 query without a usable index does, is about a hundred times cheaper than the pipeline, and the matching index brings that query down to 13 buffers. Planning stays in the low milliseconds everywhere, so the whole difference is execution.

### 5.6 Reflection

**Most expensive feature.** The rolling 30-day AOV. Alone it takes about three times as long as the order totals it builds on (5.4). The plan shows why: its two window functions need different orders, so all 99,441 orders go through a `Sort` that spills to disk (`external merge  Disk: 9440kB`), a `WindowAgg`, an `Incremental Sort` and a second `WindowAgg`. Joins are not the bottleneck: the category stage's hash joins take about 0.1 s. The other cost driver is stage 1 itself, a `Merge Join` over two `Index Scan`s that touches 100,347 buffers for 99,441 orders.

**Cache it?** Yes. The full pipeline takes seconds and spills to temporary files, while a lookup in the view is an `Index Scan` over 8 buffers in a fraction of a millisecond. Olist is historical, so an on-demand refresh after each data load is enough. With a live feed, a nightly `REFRESH ... CONCURRENTLY` fits features measured in days, such as recency or a 30-day average.

**When a plain VIEW is better.** When freshness matters more than speed and the query is cheap or selective, such as today's orders on a dashboard or a feature an online model needs right after a cancellation. Section 4.4 showed the view still reporting 0 while the table already said canceled.

**Termination.** Each step joins only the rows of the previous step; the recursion stops when a step returns nothing. A cycle A → B → A would produce rows forever: `NOT next_id = ANY(path_ids)` stops at the first repeat, and `depth < 50` is the backstop. With the depth guard alone, the walk produced 230 rows for 10 customers. The `CYCLE` clause does the same bookkeeping automatically.

**Key trade-off.** Recompute cost against freshness. Storage is small (one row per customer), and the index solves latency.

## 6. Final checks

In [41]:
def scalar(query: str):
    return q(query).iloc[0, 0]


assert scalar("SELECT current_setting('server_version_num')::INT") >= 160000

for raw_table in files:
    typed_table = raw_table.removesuffix("_raw")
    assert scalar(f"SELECT COUNT(*) FROM {raw_table}") == scalar(f"SELECT COUNT(*) FROM {typed_table}")

n_customers = scalar("SELECT COUNT(*) FROM olist_customers")
assert scalar("SELECT COUNT(*) FROM mv_olist_customer_features") == n_customers
assert scalar("SELECT COUNT(DISTINCT customer_id) FROM mv_olist_customer_features") == n_customers

unique_index = q("""
    SELECT i.indisunique, i.indpred IS NULL AS not_partial
    FROM pg_index AS i
    WHERE i.indexrelid = 'idx_mv_olist_customer_features_customer_id'::regclass
""").iloc[0]
assert unique_index["indisunique"] and unique_index["not_partial"]
assert scalar("SELECT to_regclass('idx_mv_olist_customer_features_monetary') IS NOT NULL")
assert scalar("SELECT to_regclass('hw5_mv_refresh_demo') IS NULL")

assert scalar("SELECT COUNT(*) FROM hw5_referrals") == 9
assert scalar("SELECT COUNT(*) FROM olist_orders WHERE order_status = 'canceled'") == scalar(
    "SELECT COUNT(*) FROM olist_orders_raw WHERE order_status = 'canceled'")
assert scalar("""
    SELECT COUNT(*) FROM mv_olist_customer_features AS mv
    JOIN olist_orders AS o ON o.customer_id = mv.customer_id
    WHERE mv.cancellation_rate <> CASE WHEN o.order_status = 'canceled' THEN 1 ELSE 0 END
""") == 0

lookup = explain_runs["view: lookup by customer_id"]
full = explain_runs["pipeline: all stages (view definition)"]
assert lookup["seq_scans"] == 0 and lookup["index_scans"] >= 1
assert lookup["execution_ms"] < full["execution_ms"]

print(f"customers: {n_customers:,} | view rows: {n_customers:,} | referrals: 9")
print("All checks passed")

customers: 99,441 | view rows: 99,441 | referrals: 9
All checks passed
